Importation des bibliothèque

In [1]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt 
from biopandas.pdb import PandasPdb
import nglview as nv

In [2]:
def view_structure(file_pdb):
    # 1. Charger le fichier généré
    view = nv.show_file(file_pdb)
    # 2. Vider les représentations par défaut
    view.clear_representations()
    # 3. Afficher toute la molécule de manière fine
    view.add_representation('licorice', selection='all', color='element')
    # 4. Mettre en surbrillance spéciale les atomes de Phosphore (les "noeuds" des liaisons phosphodiester)
    view.add_representation('spacefill', selection='_P', color='orange', radius=0.8)
    # 5. Rajouter un "tube" qui suit et met en évidence de façon continue le squelette phosphodiester (backbone)
    view.add_representation('tube', selection='backbone', color='red', radius=0.2)
    
    # # 6. Ajout d'un dégradé de couleur pour voir l'orientation 5' -> 3'
    # # Le schéma 'resindex' colore du bleu (début/5') au rouge (fin/3')
    # view.add_representation('cartoon', selection='nucleic', color='resindex')
    
    # 7. Centrer la vue
    view.center()
    # Afficher l'interface
    return view

Création de la structure 3D 

**AmberTools** (qui fait partie du logiciel plus large **AMBER** - *Assisted Model Building with Energy Refinement*) est l'une des suites logicielles les plus célèbres et puissantes au monde en bioinformatique structurale et en chimie numérique. 

Nous avons utilisé un de ses modules clés : **`tleap`**.

### 1. Qu'est-ce qu'AMBER / AmberTools ?
C'est un ensemble de programmes conçus à l'origine pour faire de la **Dynamique Moléculaire (MD)**, c'est-à-dire simuler informatiquement comment les atomes d'une molécule (comme l'ADN, l'ARN, ou les protéines) bougent et interagissent au cours du temps selon les lois de la physique.

Pour que ces simulations soient réalistes, AMBER utilise ce qu'on appelle des **"Champs de force" (Force Fields)**. 

### 2. Le rôle crucial du champ de force (`leaprc.RNA.OL3`)
Un champ de force est une énorme base de données mathématique élaborée par des chercheurs au fil des décennies. Dans le script, la ligne `source leaprc.RNA.OL3` charge le champ de force "OL3" (créé spécifiquement pour l'ARN). Ce fichier contient des règles strictes sur la forme que doit prendre chaque nucléotide :
*   **Les distances exactes** (en Ångströms, $10^{-10}$ m) : par exemple, la distance exacte que doit avoir la liaison covalente entre le Phosphore et l'Oxygène dans une liaison phosphodiester.
*   **Les angles de liaison** : l'angle mathématique exact que forment 3 atomes (ex: Oxygène-Phosphore-Oxygène).
*   **Les angles dièdres (torsion)** : la façon dont le squelette sucre-phosphate de l'ARN a le droit de "tourner" ou de se vriller dans l'espace 3D.
*   **Les charges électriques partelles** de chaque atome.

### 3. Comment `tleap` agit sur ta séquence
Quand on donne à `tleap` la séquence `A U G C G A U U U A C`, `tleap` agit comme un "constructeur de Lego moléculaire" ultra-précis :
1.  Il va chercher dans sa bibliothèque à quoi ressemble un 'A' (Adénine), un 'U' (Uracile), etc. en 3D (avec tous leurs atomes de carbone, hydrogène, azote, etc.).
2.  Il les attache les uns aux autres **en créant mathématiquement les liaisons phosphodiester** (le fameux pont P-O).
3.  Pour éviter que les atomes ne se rentrent dedans et pour respecter les équations de la physique spatiale du champ de force OL3, il arrange automatiquement toute la chaîne dans la position d'équilibre thermodynamique standard pour l'ARN : **l'Hélice A droite (A-form helix)**.
4.  Enfin, il exporte les coordonnées (X, Y, Z) point par point de chaque atome généré dans le fichier PDB (`.pdb` pour Protein Data Bank), qui est le format universel lisible par des logiciels comme `nglview` ou PyMOL.

In [ ]:
import os
import subprocess

def generer_arn_droit(sequence, fichier_sortie="arn_structure.pdb"):
    """
    Génère un fichier PDB d'un ARN simple brin droit (canonique) à partir 
    d'une séquence de nucléotides.
    Nécessite AmberTools ('tleap' installé et dans le PATH).
    """
    # 1. Formater la séquence pour tleap
    # Avec leaprc.RNA.OL3, les nucléotides sont juste A, C, G, U (séparés par un espace)
    sequence_formatee = " ".join([nuc.upper() for nuc in sequence])
    
    script_tleap = "instructions_tleap.in"
    
    # 2. Créer le script d'instructions que l'on passera à tleap
    with open(script_tleap, "w") as f:
        # Charger le champ de force pour l'ARN (OL3 est le standard très recommandé)
        f.write("source leaprc.RNA.OL3\n")
        
        # Construire la séquence et l'associer à une variable "mon_arn"
        f.write(f"mon_arn = sequence {{ {sequence_formatee} }}\n")
        
        # Exporter la molécule au format PDB
        f.write(f"savepdb mon_arn {fichier_sortie}\n")
        
        # Quitter le programme
        f.write("quit\n")
        
    print(f"Génération de la structure pour la séquence : {sequence}")
    
    # 3. Lancer tleap depuis Python en tant que sous-processus
    try:
        process = subprocess.run(c
            ["tleap", "-f", script_tleap], 
            stdout=subprocess.PIPE, 
            stderr=subprocess.PIPE, 
            check=True,
            text=True
        )
        print(f"✅ Succès ! Le fichier '{fichier_sortie}' a été créé.")
        
    except FileNotFoundError:
        print("❌ Erreur : 'tleap' est introuvable.")
        print("Assure-toi d'avoir activé ton environnement avec conda : conda activate Stage")
    except subprocess.CalledProcessError as e:
        print("❌ Erreur lors de l'exécution de tleap !")
        print("Détails stdout :", e.stdout)
        print("Détails stderr :", e.stderr)
        
    finally:
        # 4. Nettoyer les fichiers temporaires générés par AmberTools
        if os.path.exists(script_tleap):
            os.remove(script_tleap)
        if os.path.exists("leap.log"):
            os.remove("leap.log")

# ==========================================
# Exemple d'utilisation
# ==========================================
sequence_arn = "CCUGGUAUUGCAGUACCUCCAGGU"
generer_arn_droit(sequence_arn, "test.pdb")


Génération de la structure pour la séquence : CCUGGUAUUGCAGUACCUCCAGGU
✅ Succès ! Le fichier 'test.pdb' a été créé.


Récupération du fichier PDB sans les Hydrogène

In [4]:
ppdb_df =  PandasPdb().read_pdb('mon_arn_droit.pdb')
atom_df = ppdb_df.df["ATOM"]
atom_sans_h_df = atom_df[~atom_df["atom_name"].str.startswith("H")]
atom_sans_h_df.head(20)
ppdb_df.df['ATOM'] = atom_sans_h_df
ppdb_df.to_pdb('structure_droite_sans_h.pdb')

In [10]:
view_structure("test.pdb")

NGLWidget()

In [8]:
view_structure("fichier_arn/1A9N.pdb")

NGLWidget()

In [10]:
view_structure("resultat/mon_arn_rigide_optimise_1.pdb")

NGLWidget()

In [4]:
view_structure("resultat/structure_complete_optimisee.pdb")

NGLWidget()

In [3]:
import nglview as nv
from ipywidgets import GridspecLayout

# Imaginons que vous ayez une liste de fichiers PDB
files = ["resultat/mon_arn_optimise_24_dfire.pdb", "resultat/mon_arn_optimise_57_dfire.pdb", "resultat/mon_arn_optimise_70_dfire.pdb", "resultat/mon_arn_optimise_127_dfire.pdb", "resultat/mon_arn_optimise_155_dfire.pdb"]
views = []

for f in files:
    v = nv.show_file(f)
    v.clear()
    v.add_representation('cartoon', selection='nucleic', color='seagreen')
    v.add_representation('base', selection='nucleic', color='seagreen')
    # On réduit la taille pour que ça tienne dans la grille
    v.layout.height = '300px'
    v.layout.width = '300px'
    views.append(v)

# Création d'une grille 2x2
grid = GridspecLayout(3, 2)
for i, v in enumerate(views):
    grid[i // 2, i % 2] = v

grid

GridspecLayout(children=(NGLWidget(layout=Layout(grid_area='widget001', height='300px', width='300px')), NGLWi…

In [32]:
import nglview as nv
from ipywidgets import GridspecLayout

# Imaginons que vous ayez une liste de fichiers PDB
files = ["resultat/mon_arn_optimise_rasp_24.pdb", "resultat/mon_arn_optimise_rasp_57.pdb", "resultat/mon_arn_optimise_rasp_70.pdb", "resultat/mon_arn_optimise_rasp_127.pdb", "resultat/mon_arn_optimise_rasp_155.pdb"]
views = []

for f in files:
    v = nv.show_file(f)
    v.clear()
    v.add_representation('cartoon', selection='nucleic', color='seagreen')
    v.add_representation('base', selection='nucleic', color='seagreen')
    # On réduit la taille pour que ça tienne dans la grille
    v.layout.height = '300px'
    v.layout.width = '300px'
    views.append(v)

# Création d'une grille 2x2
grid = GridspecLayout(3, 2)
for i, v in enumerate(views):
    grid[i // 2, i % 2] = v

grid

GridspecLayout(children=(NGLWidget(layout=Layout(grid_area='widget001', height='300px', width='300px')), NGLWi…

In [6]:
import nglview as nv
from ipywidgets import GridspecLayout

# Imaginons que vous ayez une liste de fichiers PDB
files = ["resultat/arn_c3_optimisee_24.pdb", "resultat/arn_c3_optimisee_57.pdb", "resultat/arn_c3_optimisee_70.pdb", "resultat/arn_c3_optimisee_127.pdb", "resultat/arn_c3_optimisee_155.pdb"]
views = []

for f in files:
    v = nv.show_file(f)
    v.clear()
    v.add_representation('licorice', selection='all', color='element')
    # 4. Mettre en surbrillance spéciale les atomes de Phosphore (les "noeuds" des liaisons phosphodiester)
    v.add_representation('spacefill', selection='_P', color='orange', radius=0.8)
    # 5. Rajouter un "tube" qui suit et met en évidence de façon continue le squelette phosphodiester (backbone)
    v.add_representation('tube', selection='backbone', color='red', radius=0.2)
    # On réduit la taille pour que ça tienne dans la grille
    v.layout.height = '300px'
    v.layout.width = '300px'
    views.append(v)

# Création d'une grille 2x2
grid = GridspecLayout(3, 2)
for i, v in enumerate(views):
    grid[i // 2, i % 2] = v

grid

GridspecLayout(children=(NGLWidget(layout=Layout(grid_area='widget001', height='300px', width='300px')), NGLWi…

In [5]:
view_structure("resultat/arn_c3_optimisee_rasp.pdb")

NGLWidget()

In [3]:
view_structure("resultat/mon_arn_optimise_rasp.pdb")

ValueError: you must provide file extension if using file-like object or text content

In [4]:
view_structure("resultat/optimized_rasp_1773995034.pdb")

NGLWidget()